# sklearn & train_test_split — Explained

## What is sklearn?

**Scikit-learn (sklearn)** is Python's most popular **machine learning utility library**. It is not for building deep learning models — it is for:
- Data preprocessing
- Splitting datasets
- Evaluation metrics
- Classical ML algorithms (SVM, Random Forest, etc.)

Think of it as the **toolbox** that helps you prepare and evaluate models, regardless of whether you use TensorFlow, PyTorch, or sklearn itself.

In [ ]:
from sklearn.model_selection import train_test_split

## What is `train_test_split` doing?

It **splits your dataset into two parts** — one for training the model, one for evaluating it.

```python
x_train, x_eval, y_train, y_eval = train_test_split(
    x_data,           # your input features (independent variables)
    y_true,           # your labels/targets (dependent variable)
    test_size=0.3,    # 30% goes to eval, 70% goes to training
    random_state=101  # seed for reproducibility (same split every run)
)
```

**Visually:**
```
Full dataset (1,000,000 samples)
        ↓
┌─────────────────────┬───────────────┐
│     x_train         │    x_eval     │
│     y_train         │    y_eval     │
│   70% (700,000)     │  30% (300,000)│
│  → train the model  │ → test it     │
└─────────────────────┴───────────────┘
```

---

## Why split at all?

If you train AND test on the same data, the model just **memorizes** it — like giving a student the exam answers beforehand. The eval set is **unseen data** that honestly measures how well the model generalizes.

---

## What each variable holds:

| Variable | Contains | Size |
|---|---|---|
| `x_train` | input features for training | 700,000 samples |
| `y_train` | labels for training | 700,000 labels |
| `x_eval` | input features for evaluation | 300,000 samples |
| `y_eval` | labels for evaluation | 300,000 labels |

---

## `random_state=101`

Without this, the split is random every run — you'd get different splits each time. Setting a fixed seed ensures **reproducibility** — same split every time you run the code. The number `101` is arbitrary; any integer works.

## Understanding `x_train.shape` → `(700000,)`

```python
print(x_train.shape)
# Output: (700000,)
```

| Shape | Meaning |
|---|---|
| `(700000,)` | 1D — 700,000 scalar values, **no second dimension** |
| `(700000, 2)` | 2D — 700,000 rows, each with 2 features |
| `(700000, 10)` | 2D — 700,000 rows, each with 10 features |

The trailing comma with nothing after it means it is a **1D array** — each sample is a single value, not a row of multiple features.

> **Potential issue:** If feeding into `featCols` with `shape=(2,)`, you would need to reshape `x_train` to `(700000, 2)` first — a 1D array won't match a feature column expecting 2 features per sample.

In [ ]:
input_func = tf.estimator.inputs.numpy_input_fn(
    {'x': x_train},
    y_train,
    batch_size=8,
    num_epochs=None,
    shuffle=True
)

## `tf.estimator.inputs.numpy_input_fn` — Explained

This creates an **input function** — a wrapper that feeds your numpy data into the Estimator during training.

The Estimator doesn't accept raw arrays directly. It needs a callable function that returns data in batches. This line builds that function.

---

## Breaking down each argument:

**`{'x': x_train}`**
- A dictionary mapping the feature key to your data
- The key `'x'` **must match** the name you gave in `featCols = numeric_column('x', ...)`
- This is how the Estimator knows which data goes to which feature column

**`y_train`**
- Your labels/targets (the dependent variable)
- Passed separately from features

**`batch_size=8`**
- Instead of feeding all 700,000 samples at once, it feeds **8 samples at a time**
- Why? Training on all data at once is too memory-heavy. Batching makes it feasible
- 8 is quite small — common values are 32, 64, 128, 256

**`num_epochs=None`**
- How many times to loop through the full dataset
- `None` means **loop forever** — typically used during training (you control stopping via `steps=` in `.train()`)
- For evaluation you'd set `num_epochs=1` (go through data once)

**`shuffle=True`**
- Randomly shuffles the data each epoch so the model doesn't learn order patterns

---

## Full picture of how it connects:

```
numpy arrays (x_train, y_train)
        ↓
numpy_input_fn  ← wraps data, handles batching + shuffling
        ↓
input_func()    ← returns {'x': batch_of_8}, labels_of_8
        ↓
estimator.train(input_fn=input_func)
        ↓
LinearClassifier trains on those batches
```

The Estimator calls `input_func()` repeatedly during training, getting 8 samples each time until `num_epochs` or `steps` is exhausted.

In [ ]:
# Cell 52 - infinite, shuffled (used for training)
input_func = tf.estimator.inputs.numpy_input_fn({'x': x_train}, y_train, batch_size=8, num_epochs=None, shuffle=True)

# Cell 53 - 1000 epochs, no shuffle (used to score on train data)
train_input_func = tf.estimator.inputs.numpy_input_fn({'x': x_train}, y_train, batch_size=8, num_epochs=1000, shuffle=False)

# Cell 54 - 1000 epochs, no shuffle (used to score on eval data)
eval_input_func = tf.estimator.inputs.numpy_input_fn({'x': x_eval}, y_eval, batch_size=8, num_epochs=1000, shuffle=False)

# Cell 55 - actually trains the model
estimator.train(input_fn=input_func, steps=1000)

## Three Input Functions + Training

Three input functions are set up for different purposes, then training is kicked off.

---

### `input_func` — infinite, shuffled
```python
input_func = numpy_input_fn({'x': x_train}, y_train, batch_size=8, num_epochs=None, shuffle=True)
```
- Loops forever, shuffles data — designed to be used with `steps=` to control when to stop
- General-purpose training input

---

### `train_input_func` — 1000 epochs, no shuffle
```python
train_input_func = numpy_input_fn({'x': x_train}, y_train, batch_size=8, num_epochs=1000, shuffle=False)
```
- Uses **training data**, but `shuffle=False` — used later to **evaluate the model on training data** (to check for overfitting)
- No shuffle because you want consistent, repeatable results when scoring

---

### `eval_input_func` — 1000 epochs, no shuffle
```python
eval_input_func = numpy_input_fn({'x': x_eval}, y_eval, batch_size=8, num_epochs=1000, shuffle=False)
```
- Same as above but uses **eval/test data** — used to measure real performance on unseen data

---

### `estimator.train(input_fn=input_func, steps=1000)` — actually trains
- Feeds `input_func` into the `LinearClassifier` and runs **1000 steps** (1000 batches of 8)
- `num_epochs=None` in `input_func` means it won't run out of data before 1000 steps finish

---

## The Full Pattern:

```
input_func        → train the model      (estimator.train)
train_input_func  → score on train data  (estimator.evaluate) ← check overfitting
eval_input_func   → score on eval data   (estimator.evaluate) ← real performance
```

> Having both train and eval scores lets you compare: if train accuracy is high but eval is low, the model is **overfitting**.